In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anonymous111111111/doha-dataset/dohas_final_hindi_dataset.csv
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/adapter_model.safetensors
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/trainer_state.json
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/training_args.bin
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/adapter_config.json
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/README.md
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/tokenizer.json
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/tokenizer_config.json
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/scaler.pt
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/scheduler.pt
/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1/op

In [2]:
!pip install -q transformers datasets peft pandas accelerate sentencepiece protobuf

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import torch
import math
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

# ── Config ──────────────────────────────────────────────
base_model_name = "google/mt5-base"
model_path      = "/kaggle/input/models/anonymous111111111/mt5-base-ep3/transformers/epoch-3/1"   # e.g. /kaggle/input/mt5-base-ep3/transformers/mt5-base-ep3/1
device          = "cuda" if torch.cuda.is_available() else "cpu"

# Reference dohas for BLEU — add a few real dohas on the same theme
# Each reference is a list of tokens (characters work well for Hindi)
references = [
    list("राम नाम जप लीजिए, जब लग दम में दम।"),
    list("भक्ति करो भगवान की, मन में रखो विश्वाम।"),
]

# ── Load model ───────────────────────────────────────────
print("Loading model...")
tokenizer  = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
model      = PeftModel.from_pretrained(base_model, model_path)
model.to(device)
model.eval()
print("Model loaded!\n")

# ── Prompt ───────────────────────────────────────────────
theme      = "भक्ति"
context    = "ईश्वर के प्रति समर्पण"
input_text = f"Generate Hindi Doha | Theme: {theme} | Context: {context}"
input_ids  = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

print("=" * 60)
print(f"Input Prompt: {input_text}")
print("=" * 60)

# ── Generate ─────────────────────────────────────────────
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_length=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.5
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nGenerated Doha:\n{result}\n")

# ── Perplexity ───────────────────────────────────────────
# Perplexity measures how "surprised" the model is by its own output.
# Lower = more confident and fluent. Computed as exp(cross-entropy loss).
print("-" * 60)
print("Computing Perplexity...")

result_ids = tokenizer(result, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    loss = model(
        input_ids=input_ids,
        labels=result_ids
    ).loss

perplexity = math.exp(loss.item())
print(f"Perplexity : {perplexity:.4f}   (lower is better)")

# ── BLEU Score ───────────────────────────────────────────
# BLEU compares the generated doha against reference dohas.
# Score is between 0 and 1. Higher = closer to reference style.
# We use character-level tokens since Hindi words don't split on spaces cleanly.
print("-" * 60)
print("Computing BLEU Score...")

hypothesis = list(result)   # character-level tokens
smoother   = SmoothingFunction().method1

bleu = sentence_bleu(
    references,
    hypothesis,
    smoothing_function=smoother
)
print(f"BLEU Score : {bleu:.4f}   (higher is better, max 1.0)")

# ── Summary ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Generated : {result}")
print(f"  Perplexity: {perplexity:.4f}")
print(f"  BLEU Score: {bleu:.4f}")
print("=" * 60)

Loading model...


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded!

Input Prompt: Generate Hindi Doha | Theme: भक्ति | Context: ईश्वर के प्रति समर्पण

Generated Doha:
कल्याण की नई भक्ति, सभी के प्रति समर्पण, प्रकार माँवति का. उसके प्रति समर्पपण तत्त्व को ॥

------------------------------------------------------------
Computing Perplexity...
Perplexity : 14.4701   (lower is better)
------------------------------------------------------------
Computing BLEU Score...
BLEU Score : 0.1100   (higher is better, max 1.0)

SUMMARY
  Generated : कल्याण की नई भक्ति, सभी के प्रति समर्पण, प्रकार माँवति का. उसके प्रति समर्पपण तत्त्व को ॥
  Perplexity: 14.4701
  BLEU Score: 0.1100


In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import math
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

nltk.download('punkt', quiet=True)

# ── Config ──────────────────────────────────────────────
model_name = "google/mt5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Reference dohas for BLEU
references = [
    list("राम नाम जप लीजिए जब लग दम में दम"),
    list("भक्ति करो भगवान की मन में रखो विश्वास")
]

# ── Load Pretrained Base Model ──────────────────────────
print("Loading base mT5 model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.to(device)
model.eval()

print("Base model loaded!\n")

# ── Prompt ───────────────────────────────────────────────
theme = "भक्ति"
context = "ईश्वर के प्रति समर्पण"

input_text = f"Generate Hindi Doha | Theme: {theme} | Context: {context}"

input_ids = tokenizer(
    input_text,
    return_tensors="pt"
).input_ids.to(device)

print("=" * 60)
print(f"Input Prompt: {input_text}")
print("=" * 60)

# ── Generate Doha ────────────────────────────────────────
with torch.no_grad():

    outputs = model.generate(
        input_ids=input_ids,
        max_length=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.5
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nGenerated Doha:\n")
print(result)

# ── Perplexity Calculation ──────────────────────────────
print("\n" + "-" * 60)
print("Computing Perplexity...")

result_ids = tokenizer(
    result,
    return_tensors="pt"
).input_ids.to(device)

with torch.no_grad():

    loss = model(
        input_ids=input_ids,
        labels=result_ids
    ).loss

perplexity = math.exp(loss.item())

print(f"Perplexity : {perplexity:.4f}   (lower is better)")

# ── BLEU Score ──────────────────────────────────────────
print("-" * 60)
print("Computing BLEU Score...")

hypothesis = list(result)

smoother = SmoothingFunction().method1

bleu = sentence_bleu(
    references,
    hypothesis,
    smoothing_function=smoother
)

print(f"BLEU Score : {bleu:.4f}   (higher is better, max 1.0)")

# ── Summary ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Generated : {result}")
print(f"Perplexity: {perplexity:.4f}")
print(f"BLEU Score: {bleu:.4f}")

print("=" * 60)

Loading base mT5 model...


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Base model loaded!

Input Prompt: Generate Hindi Doha | Theme: भक्ति | Context: ईश्वर के प्रति समर्पण

Generated Doha:

<extra_id_0> for a simple...

------------------------------------------------------------
Computing Perplexity...
Perplexity : 10.4723   (lower is better)
------------------------------------------------------------
Computing BLEU Score...
BLEU Score : 0.0077   (higher is better, max 1.0)

SUMMARY
Generated : <extra_id_0> for a simple...
Perplexity: 10.4723
BLEU Score: 0.0077
